In [32]:
%useLatestDescriptors
%use dataframe
%use kandy

In [20]:
val corriente = DataFrame.readCsv("Corriente.csv", delimiter = ';')
val tension = DataFrame.readCsv("Tension.csv", delimiter = ';')

In [21]:
val reference_i = corriente.get { ic }.last()
val nCorriente = corriente.update { ic }.with { it / reference_i }
nCorriente.writeCsv("corriente_escalada.csv")

val reference_v = tension.get { vce }.first() ?: 1.0
val nTension = tension.update { vce }.with { if(it!=null) it / reference_v else 1.0 }
nTension.writeCsv("tension_escalada.csv")



In [40]:
fun interp1Linear(
    x: DoubleArray,
    y: DoubleArray,
    xi: DoubleArray,
    extrapolate: Boolean = false
): DoubleArray {
    require(x.size == y.size) { "x and y must have the same length" }
    require(x.size >= 2) { "At least two data points are required" }

    // Ensure x is increasing
    val (xs, ys) = if (x[0] <= x.last()) {
        x to y
    } else {
        x.reversedArray() to y.reversedArray()
    }

    // Helper: linear interpolation between two points
    fun lerp(x1: Double, y1: Double, x2: Double, y2: Double, xq: Double): Double {
        return y1 + (xq - x1) * (y2 - y1) / (x2 - x1)
    }

    val result = DoubleArray(xi.size)

    for (i in xi.indices) {
        val xq = xi[i]

        val idx = xs.binarySearch(xq)
        if (idx >= 0) {
            // Exact match
            result[i] = ys[idx]
            continue
        }

        val ins = -idx - 1
        when {
            ins == 0 -> { // Before first point
                result[i] = if (extrapolate) lerp(xs[0], ys[0], xs[1], ys[1], xq) else 0.0
            }
            ins >= xs.size -> { // After last point
                result[i] = if (extrapolate) lerp(xs[xs.size - 2], ys[ys.size - 2], xs.last(), ys.last(), xq) else 0.0
            }
            else -> { // Inside range
                result[i] = lerp(xs[ins - 1], ys[ins - 1], xs[ins], ys[ins], xq)
            }
        }
    }

    return result
}

Energia total de la grafica: 0,0000001448706351


In [41]:
val pointCount = 100
val deltaT = 2000e-9 / pointCount
val interpT = DoubleArray(pointCount) { i -> i * deltaT}
val interpVce = interp1Linear(
    x = nTension.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nTension.getColumn { vce }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val interpIc = interp1Linear(
    x = nCorriente.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nCorriente.getColumn { ic }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val pInst = interpVce.zip(interpIc) { vce, ic -> vce * ic }
val E = deltaT / 2 * (pInst.first() + 2 * pInst.drop(1).dropLast(1).fold(0.0) { acc, v -> acc + v} + pInst.last())
println("Energia total de la grafica: ${String.format("%.16f", E)}")

Energia total de la grafica: 0,0000001448706351


In [38]:
run {
    val data = mapOf(
        "tiempo" to interpT.toList() + interpT.toList() /*+ interpT.toList()*/,
        "y" to interpIc.toList() + interpVce.toList() /*+ pInst.toList()*/,
        "legends" to List(interpT.size) { "Ic" } + List(interpT.size) { "Vce" }// + List(interpT.size) { "P" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("tiempo") { axis.name = "Corriente de Pico [A]"}
                y("y") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="UjE1s0"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"tiempo",
"y":"y",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce"],
"legends":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce"],
"tiempo":[0.0,2.0E-8,4.0E-8,6.000000000000001E-8,8.0E-8,1.0E-7,1.2000000000000002E-7,1.4E-7,1.6E-7,1.8E-7,2.0E-7,2.2E-7,2.4000000000000003E-7,2.6E-7,2.8E-7,3.0E-7,3.2E-7,3.4000000000000003E-7,3.6E-7,3.8E-7,4.0E-7,4.2E-7,4.4E-7,4.6E-7,4.800000000000001E-7,5.0E-7,5.2E-7,5.4E-7,5.6E-7,5.800000000000001E-7,6.0E-7,6.2E-7,6.4E-7,6.6E-7,6.800000000000001E-7,7.0E-7,7.2E-7,7.4E-7,7.6E-7,7.8E-7,8.0E-7,8.2E-7,8.4E-7,8.6E-7,8.8E-7,9.000000000000001E-7,9.2E-7,9.4E-7,9.600000000000001E-7,9.8E-7,1.0E-6,1.02E-6,1.04E-6,1.06E-6,1.08E-6,1.1E-6,1.12E-6,1.14E-6,1.1600000000000001E-6,1.18E-6,1.2E-6,1.22E-6,1.24E-6,1.26E-6,1.28E-6,1.3E-6,1.32E-6,1.34E-6,1.3600000000000001E-6,1.3800000000000001E-6,1.4E-6,1.42E-6,1.44E-6,1.46E-6,1.48E-6,1.5E-6,1.52E-6,1.54E-6,1.56E-6,1.5800000000000001E-6,1.6E-6,1.62E-6,1.64E-6,1.66E-6,1.68E-6,1.7E-6,1.72E-6,1.74E-6,1.76E-6,1.7800000000000001E-6,1.8000000000000001E-6,1.82E-6,1.84E

In [42]:
val corriente = DataFrame.readCsv("Corriente_Apagado.csv", delimiter = ';')
val tension = DataFrame.readCsv("Tension_Apagado.csv", delimiter = ';')

In [47]:
val reference_i = corriente.get { ic }.first()
val nCorriente = corriente.update { ic }.with { it / reference_i }
nCorriente.writeCsv("corriente_escalada_apagado.csv")

val reference_v = tension.get { vce }.last()
val nTension = tension.update { vce }.with { it / reference_v }
nTension.writeCsv("tension_escalada_apagado.csv")

In [48]:
val pointCount = 100
val deltaT = 2000e-9 / pointCount
val interpT = DoubleArray(pointCount) { i -> i * deltaT}
val interpVce = interp1Linear(
    x = nTension.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nTension.getColumn { vce }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val interpIc = interp1Linear(
    x = nCorriente.getColumn { t }.toTypedArray().filterNotNull().toDoubleArray(),
    y =  nCorriente.getColumn { ic }.toTypedArray().filterNotNull().toDoubleArray(),
    xi = interpT
)
val pInst = interpVce.zip(interpIc) { vce, ic -> vce * ic }
val E = deltaT / 2 * (pInst.first() + 2 * pInst.drop(1).dropLast(1).fold(0.0) { acc, v -> acc + v} + pInst.last())
println("Energia total de la grafica: ${String.format("%.16f", E)}")

Energia total de la grafica: 0,0000002921576001


In [49]:
run {
    val data = mapOf(
        "tiempo" to interpT.toList() + interpT.toList() /*+ interpT.toList()*/,
        "y" to interpIc.toList() + interpVce.toList() /*+ pInst.toList()*/,
        "legends" to List(interpT.size) { "Ic" } + List(interpT.size) { "Vce" }// + List(interpT.size) { "P" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("tiempo") { axis.name = "Corriente de Pico [A]"}
                y("y") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="cX5wfr"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"tiempo",
"y":"y",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce"],
"legends":["Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Ic","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce","Vce"],
"tiempo":[0.0,2.0E-8,4.0E-8,6.000000000000001E-8,8.0E-8,1.0E-7,1.2000000000000002E-7,1.4E-7,1.6E-7,1.8E-7,2.0E-7,2.2E-7,2.4000000000000003E-7,2.6E-7,2.8E-7,3.0E-7,3.2E-7,3.4000000000000003E-7,3.6E-7,3.8E-7,4.0E-7,4.2E-7,4.4E-7,4.6E-7,4.800000000000001E-7,5.0E-7,5.2E-7,5.4E-7,5.6E-7,5.800000000000001E-7,6.0E-7,6.2E-7,6.4E-7,6.6E-7,6.800000000000001E-7,7.0E-7,7.2E-7,7.4E-7,7.6E-7,7.8E-7,8.0E-7,8.2E-7,8.4E-7,8.6E-7,8.8E-7,9.000000000000001E-7,9.2E-7,9.4E-7,9.600000000000001E-7,9.8E-7,1.0E-6,1.02E-6,1.04E-6,1.06E-6,1.08E-6,1.1E-6,1.12E-6,1.14E-6,1.1600000000000001E-6,1.18E-6,1.2E-6,1.22E-6,1.24E-6,1.26E-6,1.28E-6,1.3E-6,1.32E-6,1.34E-6,1.3600000000000001E-6,1.3800000000000001E-6,1.4E-6,1.42E-6,1.44E-6,1.46E-6,1.48E-6,1.5E-6,1.52E-6,1.54E-6,1.56E-6,1.5800000000000001E-6,1.6E-6,1.62E-6,1.64E-6,1.66E-6,1.68E-6,1.7E-6,1.72E-6,1.74E-6,1.76E-6,1.7800000000000001E-6,1.8000000000000001E-6,1.82E-6,1.84E